In [ ]:
import pandas as pd
import numpy as np

# ==============================
# 1. LOAD DATA
# ==============================
# Load the raw dataset before any schema cleanup or yield calculation.
df = pd.read_csv('india_crop_yield.csv')

# ==============================
# 2. CLEAN COLUMN NAMES
# ==============================
# Normalize column names so later steps can rely on a stable lowercase schema.
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_')
    .str.replace(r'[^a-z0-9_]', '', regex=True)
)

print("Columns after cleaning:", df.columns.tolist())

# ==============================
# 3. IDENTIFY REQUIRED COLUMNS
# ==============================
# Find the source columns dynamically because the upstream file can vary slightly.
def find_col(keyword):
    matches = [col for col in df.columns if keyword in col]
    return matches[0] if matches else None

state_col = find_col('state')
crop_col = find_col('crop')
year_col = find_col('year')
season_col = find_col('season')
production_col = find_col('production')
area_col = find_col('area')

required = [state_col, crop_col, year_col, season_col, production_col, area_col]

if None in required:
    raise ValueError(f"Missing columns. Found: {df.columns.tolist()}")

# Rename the fields into the canonical project schema used everywhere else.
df = df.rename(columns={
    state_col: 'state',
    crop_col: 'crop',
    year_col: 'crop_year',
    season_col: 'season',
    production_col: 'production',
    area_col: 'area'
})

# ==============================
# 4. REVIEW UNIQUE CROPS
# ==============================
# Check crop diversity before cleaning the numeric fields.
print("Initial unique crops:", df['crop'].nunique())

# ==============================
# 5. CLEAN NUMERIC DATA
# ==============================
# Coerce numeric fields so invalid strings become NaN and can be filtered out.
df['production'] = pd.to_numeric(df['production'], errors='coerce')
df['area'] = pd.to_numeric(df['area'], errors='coerce')

df = df.dropna(subset=['production', 'area'])
df = df[df['area'] > 0]

# ==============================
# 6. CALCULATE YIELD
# ==============================
# Convert production and area into the yield metric used by training and inference.
df['yield_kg_ha'] = (df['production'] * 1000) / df['area']

# ==============================
# 7. REMOVE OUTLIERS
# ==============================
# Remove extreme yields so the downstream model is trained on a realistic range.
df = df[(df['yield_kg_ha'] > 10) & (df['yield_kg_ha'] < 100000)]

# ==============================
# 8. FINAL DATASET
# ==============================
# Keep only the fields used by the feature engineering notebook and app.
df = df[['state', 'crop_year', 'season', 'crop', 'yield_kg_ha']]

print("Final shape:", df.shape)
print(df.head())

# ==============================
# 9. EXPORT
# ==============================
# Persist the cleaned dataset for the next notebook in the pipeline.
df.to_csv('cleaned.csv', index=False)

print("âœ… cleaned.csv saved successfully")